# CRC Screening Tool
**Colorectal Carcinoma Screening Pipeline for Digital Pathology**

---

This notebook provides an end-to-end pipeline for screening colorectal carcinoma (CRC) from whole slide images (WSI). The pipeline uses a MobileNet-based CNN trained via Google Teachable Machine on a Kenyan colorectal carcinoma cohort from Aga Khan University Hospital (Nairobi) and Tenwek Hospital (Bomet).

**Pipeline overview:**

| Step | Description | Input | Output |
|------|-------------|-------|--------|
| 1. **View Slide** | Inspect the whole slide image before processing | `.svs` file | Visual inspection |
| 2. **Export Tiles** | Divide the WSI into 512×512 tiles at ~10× magnification | `.svs` file | JPEG tile images |
| 3. **Score Tiles** | Run each tile through the trained model | Tile folder + `.h5` model | `tile_scores.csv` |
| 4. **Evaluate** | Compute ROC curve, AUC, and summary statistics | `tile_scores.csv` | ROC figure + stats |
| 5. **Click-to-Predict** | Click on the slide to predict any region interactively | `.svs` file + `.h5` model | Live prediction |
| 6. **Case-Level Analysis** | Classify cases using top-K tile thresholding | Scored slide folders | Confusion matrix |
| 7. **Fine-Tune Model** | Extend the model with new labeled tiles | Tile folder + `.h5` model | Fine-tuned `.h5` model |

Steps 1–4 form the core pipeline for processing and evaluating individual slides. Step 5 provides an interactive tool for exploring model predictions on any region of a slide. Step 6 performs case-level classification across multiple slides to determine the optimal decision threshold. Step 7 allows end users to improve the model by fine-tuning it with locally accumulated cases — only the classification head is retrained while the convolutional backbone stays frozen.

Each step below is self-contained — run the cell to launch an interactive interface, then follow the on-screen prompts.

---

*Developed by Dr. Ulysses Balis, University of Michigan*  
*Pipeline packaging: Ye Chan Kim, University of Michigan*

In [1]:
%matplotlib inline

---

## Step 1 — View Whole Slide Image

Use this viewer to inspect your whole slide image before processing. This helps confirm you have the correct file and lets you explore the tissue at different magnifications.

**Instructions:**
1. Select your `.svs` file using the file browser
2. Click **Load Slide**
3. Use the **Zoom** slider to zoom in and out
4. Use the **X** and **Y** sliders to pan across the slide
5. The red rectangle on the thumbnail (left) shows your current viewport

> **Tip:** A sample slide (`8130.svs`) is available in `sample_data/wsi/` for testing.

In [2]:
from utils import create_viewer_ui
create_viewer_ui()

FileChooser(path='/Users/yechank/Desktop/mm/project/CRC_Screening_Tool_v3/sample_data/wsi/positive/2138813337'…

Button(button_style='success', description='Load Slide', icon='eye', layout=Layout(margin='10px 0px'), style=B…

HTML(value='<i>No slide loaded.</i>')

Output()

---

## Step 2 — Export Tiles from Whole Slide Image

This step divides the whole slide image into 512×512 pixel tiles at approximately 10× magnification. A content filter automatically discards blank tiles (background glass, out-of-focus regions) and keeps only tiles containing meaningful tissue.

**Instructions:**
1. Select the `.svs` file to process
2. Select an output folder where tiles will be saved
3. Choose whether to extract all tiles or set a custom limit
4. *(Optional)* Check the box to generate a tile location map after extraction
5. Click **Run Extraction** and wait for the progress bar to complete

**Output:**  
JPEG tile images saved to your chosen folder, named as `{slide_name}_x{x}_y{y}__{serial}.jpg`. The coordinates in the filename record each tile's position on the original slide.

> **Note:** Skip this step if you already have pre-exported tile images.

In [3]:
from utils import create_extraction_ui
create_extraction_ui()

FileChooser(path='/Users/yechank/Desktop/mm/project/CRC_Screening_Tool_v3/sample_data/wsi/positive/2138813337'…

FileChooser(path='/Users/yechank/Desktop/mm/project/CRC_Screening_Tool_v3/sample_data/wsi/positive/2138813337/…

HTML(value='<b>3. Tile Limit:</b>', layout=Layout(margin='10px 0px 0px 0px'))

RadioButtons(layout=Layout(width='80%'), options=('Extract all tiles', 'Set custom limit'), value='Extract all…

IntText(value=100, description='Max tiles:', layout=Layout(display='none', width='200px'), style=DescriptionSt…

HTML(value="<i style='color:gray; font-size:12px;'>Total candidate tiles will be shown when extraction starts.…

HTML(value='<b>3. Post-Extraction:</b>', layout=Layout(margin='0px 0px 0px 0px'))

Checkbox(value=True, description='Generate and show tile map after extraction', indent=False, layout=Layout(ma…

Button(button_style='success', description='Run Extraction', icon='play', layout=Layout(margin='20px 0px 10px …

Output()

---

## Step 3 — Score Tiles

This step runs each tile image through the trained CRC classification model. The model assigns a malignancy probability to every tile — a score between 0 (confidently benign) and 1 (confidently malignant).

**Instructions:**
1. Select the model file (`model/model.h5`)
2. Select the folder containing tile images
3. Select an output folder where the scores CSV will be saved
4. Enter an output prefix (e.g., `Run1_`) to label this run
5. Click **Run Scoring** and wait for the progress bar to complete

**Output:**  
`{prefix}tile_scores.csv` — a CSV file where each row contains:
- `file` — tile image path
- `truth` — ground truth label (if tiles are in `positive/` or `negative/` subfolders)
- `pred_label` — predicted class (Positive or Negative)
- `pred_prob` — confidence of prediction
- `p_Positive` — probability of malignancy
- `p_Negative` — probability of benign

> **Tip:** To measure accuracy, organize tiles into `positive/` and `negative/` subfolders before scoring. Sample data is available in `sample_data/tiles/`.

In [4]:
from utils import create_scoring_ui
create_scoring_ui()

FileChooser(path='/Users/yechank/Desktop/mm/project/CRC_Screening_Tool_v3/model', filename='', title='<b>1. Se…

FileChooser(path='/Users/yechank/Desktop/mm/project/CRC_Screening_Tool_v3/sample_data/tiles', filename='', tit…

FileChooser(path='/Users/yechank/Desktop/mm/project/CRC_Screening_Tool_v3/sample_data/tiles', filename='', tit…

HTML(value='<b>4. Output Prefix:</b>')

Text(value='Version1.0_', layout=Layout(width='40%'), placeholder='e.g., MyRun_')

HTML(value='<i style="color: gray;">Output file will be saved as: {prefix}tile_scores.csv inside the tile fold…

Button(button_style='success', description='Run Scoring', icon='play', layout=Layout(margin='20px 0px 10px 0px…

Output()

---

## Step 4 — Evaluate Results

This step takes the scored CSV from Step 3 and computes evaluation metrics, including an ROC curve, AUC, and summary statistics. An interactive ROC chart is displayed where you can hover over any point to see the threshold, TPR, FPR, precision, and F1 score at that operating point.

**Instructions:**
1. Select the `tile_scores.csv` file from Step 3
2. Select an output folder for the evaluation files
3. Enter an output prefix
4. Click **Run Evaluation**

**Output files:**
- `{prefix}tile_stats.txt` — summary statistics (tile count, score range, AUC)
- `{prefix}tile_roc.csv` — ROC curve data points at every threshold
- `{prefix}roc_curve.png` — publication-ready ROC curve figure

> **Note:** ROC evaluation requires ground truth labels. If your tiles were not organized in `positive/` and `negative/` subfolders during scoring, the ROC will be skipped and only basic statistics will be generated.

In [5]:
from utils import create_evaluation_ui
create_evaluation_ui()

FileChooser(path='/Users/yechank/Desktop/mm/project/CRC_Screening_Tool_v3/sample_data/tiles', filename='', tit…

FileChooser(path='/Users/yechank/Desktop/mm/project/CRC_Screening_Tool_v3/sample_data/tiles', filename='', tit…

HTML(value='<b>3. Output Prefix:</b>')

Text(value='Version1.0_', layout=Layout(width='40%'), placeholder='e.g., MyRun_')

HTML(value='<i style="color: gray;">Output files: {prefix}tile_stats.txt, {prefix}tile_roc.csv, {prefix}roc_cu…

Button(button_style='success', description='Run Evaluation', icon='play', layout=Layout(margin='20px 0px 10px …

Output()

---

## Step 5 — Interactive Click-to-Predict

Click anywhere on the slide thumbnail to extract a tile at that location and run it through the model. The tile and prediction update each time you click.

**Instructions:**
1. Select the model file (`model/model.h5`)
2. Select your `.svs` file
3. Click **Load Model & Slide**
4. Click anywhere on the thumbnail to predict that region

A red dot marks where you clicked, a red square shows the tile boundary, the extracted tile appears on the right, and the prediction with confidence scores appears below it.

> **Tip:** Try clicking on dark purple/pink tissue regions (likely malignant) vs pale pink/white regions (likely benign) to see how the model responds.

<img src="sample_data/wsi/opening_before_after.png" width="1000">

In [6]:
from utils import create_predict_ui
create_predict_ui()

FileChooser(path='/Users/yechank/Desktop/mm/project/CRC_Screening_Tool_v3/model', filename='', title='<b>1. Se…

FileChooser(path='/Users/yechank/Desktop/mm/project/CRC_Screening_Tool_v3/sample_data/wsi/positive/2138813337'…

Button(button_style='success', description='Load Model & Slide', icon='eye', layout=Layout(margin='10px 0px'),…

HTML(value='<i>No slide loaded.</i>')

Output()

---

## Step 6 — Case-Level Analysis (Top-K Threshold)

This step performs case-level classification using a top-K tile threshold. Given a set of scored slides organized into `positive/` and `negative/` subfolders, it determines whether each case is positive or negative based on how many tiles exceed a malignancy threshold.

**Instructions:**
1. Select the WSI directory (must contain `positive/` and `negative/` subfolders with scored slides)
2. Click **Load Cases**
3. Choose between absolute K (number of tiles) or percentage K (% of total tiles)
4. Adjust the **Tile threshold** slider to control what counts as a "malignant tile"
5. Adjust the **K** slider to find the sweet spot where FP and FN are both zero

The confusion matrix and per-case breakdown update live as you move the sliders.

> **Goal:** Find the K value where sensitivity = 1.0 and specificity = 1.0 (perfect classification).

In [7]:
from utils import create_case_analysis_ui
create_case_analysis_ui()

FileChooser(path='/Users/yechank/Desktop/mm/project/CRC_Screening_Tool_v3/sample_data/wsi', filename='', title…

HTML(value='<b>2. Scores CSV Filename:</b>')

Text(value='batch_tile_scores.csv', layout=Layout(width='40%'), placeholder='e.g., batch_tile_scores.csv')

Button(button_style='success', description='Load Cases', icon='folder-open', layout=Layout(margin='10px 0px'),…

HTML(value='<i>No cases loaded.</i>')

Output()

## Step 7: Fine-Tune Model
Select a folder of labeled tiles to extend the model. The convolutional backbone stays frozen — only the classification head is updated. The fine-tuned model can be saved and used in Steps 3–6.

In [8]:
from utils import create_finetuning_ui
create_finetuning_ui()

---

### About

| | |
|---|---|
| **Algorithm** | MobileNet-based CNN trained via Google Teachable Machine |
| **Training data** | Kenyan CRC cohort (Aga Khan University Hospital, Nairobi & Tenwek Hospital, Bomet) |
| **Tile size** | 512 × 512 px at ~10× magnification (1.0 µm/pixel) |
| **Model input** | 224 × 224 px, normalized to [-1, 1] |
| **Dependencies** | Python 3.10/3.11, TensorFlow 2.15.1, OpenSlide, Pillow, NumPy, Matplotlib, Pandas |

For questions or issues, contact the research team at the University of Michigan.